# 🐾 WILDLIFE-AI: Türkiye Yaban Hayatı Modeli Eğitimi (Google Colab / GPU)

Bu notebook ile Roboflow'daki yaban hayatı (yaban domuzu, kurt, çakal, tilki, bozayı) veri setini kullanarak **YOLO11n** modelini ücretsiz GPU (T4) ile 10-15 dakikada eğitebilirsiniz.

### Adım 1: GPU Kontrolü ve Ultralytics Kurulumu

In [ ]:
!nvidia-smi
!pip install -q ultralytics roboflow

### Adım 2: Roboflow'dan Veri Setini İndirme
*Roboflow Universe üzerinden aldığınız indirme kodunu buraya yapıştırın:*

In [ ]:
from roboflow import Roboflow
# Roboflow API anahtarınızı ve proje bilgilerinizi girin:
rf = Roboflow(api_key="BURAYA_ROBOFLOW_API_KEY_YAZIN")
project = rf.workspace("WORKSPACE_ADI").project("PROJE_ADI")
version = project.version(1)
dataset = version.download("yolov11")
print("Veri seti indirildi:", dataset.location)

### Adım 3: Transfer Learning ile YOLO11n Modelini Eğitme

In [ ]:
from ultralytics import YOLO

# Temel hafif YOLO11 nano modelini yükle
model = YOLO("yolo11n.pt")

# Eğitimi başlat (50 epoch, freeze=10 ile hızlı transfer learning)
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    freeze=10,
    device=0,
    name="wildlife_tr_model"
)

### Adım 4: Raspberry Pi 4 için ONNX Formatına Çevirme ve İndirme

In [ ]:
from google.colab import files

# En iyi model dosyasını ONNX'e çevir (Raspberry Pi 4 için)
trained_model = YOLO("runs/detect/wildlife_tr_model/weights/best.pt")
onnx_file = trained_model.export(format="onnx", imgsz=640)

# Bilgisayarınıza indirin (WILDLIFE-AI/models klasörüne atacaksınız)
files.download("runs/detect/wildlife_tr_model/weights/best.pt")
files.download(onnx_file)